# PCB Defect Inspector: training notebook (Kaggle GPU)

**Before you run:**
1. Notebook settings (right sidebar): **Accelerator = GPU T4 x1** (or P100), **Internet = On**.
2. *Add Input* -> search **`pcb-defects`** by *akhatova* (dataset `akhatova/pcb-defects`) -> Add.
3. *Add Input* -> **Upload** your project zip as a Kaggle Dataset named **`pcb-inspector-code`** (upload the unzipped `pcb-inspector` folder or the zip).

Then **Run All**. Expect roughly 60-90 minutes for 80 epochs at 1024px on a T4. Use `EPOCHS = 5` first as a dry run.

In [ ]:
EPOCHS = 80        # set to 5 for a quick dry run
IMGSZ = 1024       # PCB defects are tiny; do not go below 800 for the real run
BATCH = 8          # lower to 4 if you hit CUDA out-of-memory
MODEL = "yolov8s.pt"   # 'yolov8n.pt' is faster, 'yolov8m.pt' is more accurate but slower

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics supervision onnx onnxslim pyyaml

In [ ]:
import glob, os, shutil
cands = glob.glob('/kaggle/input/*/pcb-inspector') + glob.glob('/kaggle/input/*/*/pcb-inspector') \
      + [p for p in glob.glob('/kaggle/input/*') if os.path.exists(p + '/src/config.py')]
assert cands, "Upload the project as a Kaggle Dataset (see instructions above)."
SRC = cands[0]
shutil.rmtree('/kaggle/working/pcb-inspector', ignore_errors=True)
shutil.copytree(SRC, '/kaggle/working/pcb-inspector')
%cd /kaggle/working/pcb-inspector
!ls

## 1. Convert the dataset (VOC XML -> YOLO) and split 70/15/15

In [ ]:
!python -m src.data.prepare_dataset --src /kaggle/input --out /kaggle/working/dataset --exclude "rotat,augment,pcb-inspector-code" 

## 2. Train

In [ ]:
!python -m src.training.train --data /kaggle/working/dataset/data.yaml --model {MODEL} \
    --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --device 0

## 3. Evaluate on the held-out test split + error analysis

In [ ]:
!python -m src.training.evaluate --data /kaggle/working/dataset/data.yaml --weights models/best.pt --device 0
print(open('reports/evaluation_report.md').read())

## 4. Export to ONNX and benchmark PyTorch vs ONNX

In [ ]:
!python -m src.optimization.export_onnx --weights models/best.pt
!python -m src.optimization.benchmark --models models/best.pt models/best.onnx --images /kaggle/working/dataset/images/test --n 20 --device cpu

## 5. (Optional) Does the VLM second opinion help?
Needs a VLM. On Kaggle the easiest is `VLM_PROVIDER=mock` (pipeline check only). For a real answer run this step locally
with Ollama running (see README), or set `GEMINI_API_KEY` and `VLM_PROVIDER=gemini` here.

In [ ]:
import os
os.environ.update(VLM_PROVIDER="mock", REPORT_MODE="template", USE_EMBEDDINGS="false", DEVICE="0",
                  MODEL_PATH="/kaggle/working/pcb-inspector/models/best.pt")
!python -m src.training.evaluate_pipeline --data /kaggle/working/dataset/data.yaml --limit 30

## 6. Package the artifacts to download

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
for f in ['models/best.pt', 'models/best.onnx', 'models/model_meta.json']:
    shutil.copy(f, '/kaggle/working/artifacts/')
shutil.copytree('reports', '/kaggle/working/artifacts/reports', dirs_exist_ok=True)
shutil.copytree('docs', '/kaggle/working/artifacts/docs', dirs_exist_ok=True)
shutil.copytree('runs', '/kaggle/working/artifacts/runs', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/pcb_artifacts', 'zip', '/kaggle/working/artifacts')
print('Download /kaggle/working/pcb_artifacts.zip from the Output tab, unzip it into your local project:'
      ' models/* -> ./models, reports/* -> ./reports')